In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
-f https://data.pyg.org/whl/torch-2.9.0+cpu.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.9.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.6/669.6 kB 52.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.6/809.6 kB 52.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.2/304.2 kB 24.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import gc
import torch
from torch_geometric.data import Data
from torch_geometric.utils import add_self_loops
import numpy as np




/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/pyg_lib/libpyg.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_scatter_cpu.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_spmm_cpu.so
  import torch_geometric.typing


Paths Features Files

In [ ]:
train_pairs = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/train_pairs_random.csv")
val_pairs  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/val_pairs_random.csv")
test_pairs = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/test_pairs_random.csv")

train_paths  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/train_path_table_random.csv")
val_paths  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/val_path_table_random.csv")
test_paths = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/test_path_table_random.csv")

train_edges  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/train_edge_table_random.csv")
val_edges  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/val_edge_table_random.csv")
test_edges = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/test_edge_table_random.csv")

train_nodes  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/train_node_table_random.csv")
val_nodes  = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/val_node_table_random.csv")
test_nodes = pd.read_csv("/content/drive/MyDrive/FinalWork/Expirements/tau_0.6/test_node_table_random.csv")



In [ ]:
protein_df = pd.read_csv(
    "/content/drive/MyDrive/PROTEIN_EMBEDDINGS_8142.csv"
)

protein_ids = protein_df.iloc[:, 0].astype(str)
embeddings = protein_df.iloc[:, 1:].astype(float)

protein_emb = {
    pid: torch.tensor(embeddings.iloc[i].values, dtype=torch.float32)
    for i, pid in enumerate(protein_ids)
}

print(len(protein_emb), "protein embeddings loaded")


8142 protein embeddings loaded


In [ ]:
# Load CSV
drug_fp_df = pd.read_csv(
    "/content/drive/MyDrive/evidti_drugs_fingerprints_2048.csv", low_memory=False
)

# Fingerprint column names
fp_cols = [c for c in drug_fp_df.columns if c.startswith("fp_")]

# FORCE numeric conversion (critical)
drug_fp_df[fp_cols] = drug_fp_df[fp_cols].apply(
    pd.to_numeric, errors="coerce"
)

# Replace NaNs (very important)
drug_fp_df[fp_cols] = drug_fp_df[fp_cols].fillna(0.0)

# Build dictionary: drugbank_id -> tensor
drug_fp = {}
for _, row in drug_fp_df.iterrows():
    drug_fp[row["DrugBank_ID"]] = torch.from_numpy(
        row[fp_cols].to_numpy(dtype=np.float32)
    )

print("Loaded drug fingerprints:", len(drug_fp))
print("Fingerprint dimension:", drug_fp[next(iter(drug_fp))].shape)


Loaded drug fingerprints: 5815
Fingerprint dimension: torch.Size([2048])


In [ ]:
COMMON_DIM = 256
drug_proj = torch.nn.Linear(2048, COMMON_DIM)
protein_proj = torch.nn.Linear(1280, COMMON_DIM)

In [ ]:
protein_cache = {}

for k, v in protein_emb.items():
    protein_cache[k] = protein_proj(
        torch.as_tensor(v, dtype=torch.float32)
    ).half()

In [ ]:
drug_cache = {}
for k, v in drug_fp.items():
    drug_cache[k] = drug_proj(
        torch.as_tensor(v, dtype=torch.float32)
    ).half()

In [ ]:
print(len(drug_cache))
print(len(protein_cache))

5815
8142


This function:

Transforms a biological path into a learnable graph by enforcing semantic correctness, feature alignment, and structural validity.


In [ ]:
def build_path_graph(
    pid,
    nodes,
    edges,
    bidirectional=True
):

    if len(nodes) == 0:
        return None

    node_ids = nodes["node"].tolist()
    node_map = {nid: i for i, nid in enumerate(node_ids)}

    # ==========================
    # Node features
    # ==========================

    x = []
    node_roles = []

    for row in nodes.itertuples(index=False):

        nid = row.node
        role = row.node_role

        if role == "drug":

            node_roles.append(0)

            fp = drug_fp.get(nid)
            if fp is None:
                return None

            feat = drug_cache.get(nid)

        else:

            if role == "target":
                node_roles.append(1)
            else:
                node_roles.append(2)

            emb = protein_emb.get(nid)
            if emb is None:
                return None

            feat = protein_cache.get(nid)

        x.append(feat)

    x = torch.stack(x).to(torch.float32)

    node_roles = torch.tensor(
        node_roles,
        dtype=torch.uint8
    )

    # ==========================
    # Edges
    # ==========================

    edge_pairs = []
    edge_features = []

    for row in edges.itertuples(index=False):

        src = node_map.get(row.src)
        dst = node_map.get(row.dst)

        if src is None or dst is None:
            continue

        attr = [
            float(row.transition_prob),
            float(row.confidence_score),
            float(row.go_similarity)
        ]

        edge_pairs.append([src, dst])
        edge_features.append(attr)

        if bidirectional:
            edge_pairs.append([dst, src])
            edge_features.append(attr)

    if len(edge_pairs) == 0:

        edge_index = torch.empty(
            (2, 0),
            dtype=torch.int32
        )

        edge_attr = torch.empty(
            (0, 3),
            dtype=torch.float32
        )

    else:

        edge_index = (
            torch.tensor(edge_pairs,
                         dtype=torch.int32)
            .t()
            .contiguous()
        )

        edge_attr = torch.tensor(
            edge_features,
            dtype=torch.float32
        )

    edge_index, edge_attr = add_self_loops(
        edge_index=edge_index,
        edge_attr=edge_attr,
        fill_value=torch.zeros(
            edge_attr.size(1),
            dtype=edge_attr.dtype
        )
    )

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        node_role=node_roles
    )

In [ ]:
# normalize once
train_nodes["node"] = train_nodes["node"].astype(str)

train_edges["src"] = train_edges["src"].astype(str)
train_edges["dst"] = train_edges["dst"].astype(str)

# group once (no copies)
node_groups = train_nodes.groupby("path_id")
edge_groups = train_edges.groupby("path_id")

node_ids = set(train_nodes["path_id"].unique())
edge_ids = set(train_edges["path_id"].unique())

all_pids = sorted(node_ids & edge_ids)

In [ ]:
DRUG_ROLE = "drug"
TARGET_ROLE = "target"

train_cache = {}

for pid in all_pids:

    try:
      rows = node_groups.get_group(pid)
      edges = edge_groups.get_group(pid)
    except KeyError:
      continue
    if edges is None or rows.empty:
        continue

    # ---- role filtering ----
    drug_rows = rows[rows.node_role == DRUG_ROLE]
    target_rows = rows[rows.node_role == TARGET_ROLE]

    # ---- exactly one drug and one target ----
    if len(drug_rows) != 1 or len(target_rows) != 1:
        continue

    drug = drug_rows.iloc[0]["node"]
    target = target_rows.iloc[0]["node"]

    # ---- build graph ----
    g = build_path_graph(
        pid,
        rows,          # IMPORTANT
        edges,         # IMPORTANT
        bidirectional=True
    )

    if g is None:
        continue

    train_cache[pid] = {
        "graph": g,
        "drug": drug,
        "target": target
    }

print("Train Cached graphs:", len(train_cache))

Train Cached graphs: 515386


In [ ]:
torch.save(train_cache, "/content/drive/MyDrive/cache/train_cache_updated.pt")
del train_cache
gc.collect()


9

In [ ]:
# normalize once
val_nodes["node"] = val_nodes["node"].astype(str)

val_edges["src"] = val_edges["src"].astype(str)
val_edges["dst"] = val_edges["dst"].astype(str)

# group once
node_groups = dict(tuple(val_nodes.groupby("path_id")))
edge_groups = dict(tuple(val_edges.groupby("path_id")))

all_pids = list(node_groups.keys())

In [ ]:
DRUG_ROLE = "drug"
TARGET_ROLE = "target"

val_cache = {}

for pid in all_pids:

    rows = node_groups[pid]
    edges = edge_groups.get(pid)

    if edges is None or rows.empty:
        continue

    # ---- role filtering ----
    drug_rows = rows[rows.node_role == DRUG_ROLE]
    target_rows = rows[rows.node_role == TARGET_ROLE]

    # ---- exactly one drug and one target ----
    if len(drug_rows) != 1 or len(target_rows) != 1:
        continue

    drug = drug_rows.iloc[0]["node"]
    target = target_rows.iloc[0]["node"]

    # ---- build graph ----
    g = build_path_graph(
        pid,
        rows,          # IMPORTANT
        edges,         # IMPORTANT
        bidirectional=True
    )

    if g is None:
        continue

    val_cache[pid] = {
        "graph": g,
        "drug": drug,
        "target": target
    }

print("Val Cached graphs:", len(val_cache))

Val Cached graphs: 64192


In [ ]:
torch.save(val_cache, "/content/drive/MyDrive/cache/val_cache_updated.pt")
del val_cache
gc.collect()

9

In [ ]:
# normalize once
test_nodes["node"] = test_nodes["node"].astype(str)

test_edges["src"] = test_edges["src"].astype(str)
test_edges["dst"] = test_edges["dst"].astype(str)

# group once
node_groups = dict(tuple(test_nodes.groupby("path_id")))
edge_groups = dict(tuple(test_edges.groupby("path_id")))



all_pids = list(node_groups.keys())

In [ ]:
DRUG_ROLE = "drug"
TARGET_ROLE = "target"

test_cache = {}

for pid in all_pids:

    rows = node_groups[pid]
    edges = edge_groups.get(pid)

    if edges is None or rows.empty:
        continue

    # ---- role filtering ----
    drug_rows = rows[rows.node_role == DRUG_ROLE]
    target_rows = rows[rows.node_role == TARGET_ROLE]

    # ---- exactly one drug and one target ----
    if len(drug_rows) != 1 or len(target_rows) != 1:
        continue

    drug = drug_rows.iloc[0]["node"]
    target = target_rows.iloc[0]["node"]

    # ---- build graph ----
    g = build_path_graph(
        pid,
        rows,          # IMPORTANT
        edges,         # IMPORTANT
        bidirectional=True
    )

    if g is None:
        continue

    test_cache[pid] = {
        "graph": g,
        "drug": drug,
        "target": target
    }

print("Test Cached graphs:", len(test_cache))

Test Cached graphs: 65904


In [ ]:
torch.save(test_cache, "/content/drive/MyDrive/cache/test_cache_updated.pt")
del test_cache
gc.collect()

9